# Review 2 static training-label poisoning comparison

Import this notebook into Kaggle, enable **Internet** and a **GPU (T4)**, and attach the private `cic-collection.parquet` as a Kaggle input. Run top to bottom. This is a balanced-subset experiment on a combined flow collection with unverified per-row source provenance. It is **not** replay poisoning, continual learning, a timing backdoor, or final/full-dataset performance. Outputs are intentionally blank until Kaggle executes the notebook.


## 1. Pin the published implementation

Clone the unified feature branch and detach at the full source SHA that includes both poisoning modes and shared evaluation. Do not substitute `main`.


In [ ]:
from pathlib import Path
import subprocess, sys
SOURCE_BRANCH = 'feature/unified-novelty'
SOURCE_SHA = '316b817b40f4008af128489ac958d3b232b56049'
REPO_DIR = Path('/kaggle/working/poison-resilient-continual-nids-poisoning')
if REPO_DIR.exists():
    raise RuntimeError('Repository folder already exists; use a fresh session or inspect it before rerunning.')
subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, '--single-branch', 'https://github.com/DevChi1l/poison-resilient-continual-nids.git', str(REPO_DIR)], check=True)
subprocess.run(['git', 'checkout', '--detach', SOURCE_SHA], cwd=REPO_DIR, check=True)
checked_out_sha = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO_DIR, text=True).strip()
assert checked_out_sha == SOURCE_SHA
sys.path.insert(0, str(REPO_DIR))
print('Verified source SHA:', checked_out_sha)


## 2. Check the environment and functional APIs

Keep Kaggle's CUDA-enabled PyTorch; do not blindly install local requirements over it. These small assertions verify imports and label-flip behavior, not NIDS performance.


In [ ]:
from importlib.metadata import version
from dataclasses import asdict
from datetime import datetime, timezone
from time import perf_counter
import hashlib, json, platform
import numpy as np
import pandas as pd
import pyarrow as pa
import torch
from src.data import sample_class_balanced_indices, stratified_split_indices, prepare_sampled_dataset
from src.models import ModelConfig, TabularTransformerClassifier
from src.poisoning import apply_label_flip
from src.evaluation import classification_metrics
from src.novelty import detect_unknown
if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required. Enable Kaggle GPU and restart the session.')
assert pa.array([1, 2]).to_pylist() == [1, 2]
assert pd.Series([1, 2]).sum() == 3
assert torch.tensor([1.0], device='cuda').is_cuda
probe = apply_label_flip(np.array([0, 1, 2], dtype=np.int64), 1.0, 42, allowed_class_ids=[0, 1, 2])
assert probe.changed_count == 3 and np.all(probe.poisoned_labels != probe.original_labels)
environment = {'python': platform.python_version(), 'packages': {name: version(name) for name in ('numpy', 'pandas', 'pyarrow', 'torch')}, 'torch_cuda_build': torch.version.cuda, 'gpu_name': torch.cuda.get_device_name(0)}
print('Environment:', environment, 'functional label-flip check: passed')


## 3. Locate and identify the attached collection

Edit `PARQUET_PATH` if multiple matching files are attached. The checksum must match the reviewed clean-baseline input; hashing reads the entire file. The collection's per-row source provenance remains unverified.


In [ ]:
PARQUET_PATH = None  # e.g. '/kaggle/input/my-dataset/cic-collection.parquet'
EXPECTED_INPUT_SHA256 = '666af53c788c79312b421c607289cb555c2335ae51ba39b2700a357595fa8dba'
candidates = sorted(Path('/kaggle/input').rglob('cic-collection.parquet'))
print('Candidates:', [str(path) for path in candidates])
if PARQUET_PATH is None:
    if len(candidates) != 1:
        raise RuntimeError(f'Expected one attached Parquet file, found {len(candidates)}; set PARQUET_PATH explicitly.')
    parquet_path = candidates[0]
else:
    parquet_path = Path(PARQUET_PATH)
    if not parquet_path.is_file() or not parquet_path.is_relative_to(Path('/kaggle/input')):
        raise ValueError('PARQUET_PATH must be an attached file under /kaggle/input')
start = perf_counter()
digest = hashlib.sha256()
with parquet_path.open('rb') as source_file:
    for chunk in iter(lambda: source_file.read(8 * 1024 * 1024), b''):
        digest.update(chunk)
input_sha256 = digest.hexdigest()
checksum_seconds = perf_counter() - start
print('Input SHA-256:', input_sha256, 'bytes:', parquet_path.stat().st_size)
if input_sha256 != EXPECTED_INPUT_SHA256:
    raise RuntimeError('Input differs from reviewed clean-baseline collection; inspect before running.')


## 4. Sample, split, and preprocess exactly once

The existing seeded sampler takes up to 2,000 rows per broad class, but still scans the collection. The 70/15/15 stratified split and fitted preprocessing are shared unchanged by every condition; preprocessing fits **training rows only**. Validation and test labels remain clean.


In [ ]:
stage_start = perf_counter()
sample = sample_class_balanced_indices(parquet_path, per_class_limit=2000, seed=42)
sampling_seconds = perf_counter() - stage_start
stage_start = perf_counter()
split = stratified_split_indices(sample.labels, seed=42)
dataset = prepare_sampled_dataset(parquet_path, sample, split)
preparation_seconds = perf_counter() - stage_start
for name in ('train', 'validation', 'test'):
    part = getattr(dataset, name)
    assert part.features.dtype == np.float32 and part.labels.dtype == np.int64
    assert part.features.shape[1] == 54 and np.isfinite(part.features).all()
    print(name, part.features.shape, 'finite:', True)
print('Class mapping:', dataset.class_names, 'available:', sample.available_counts)
class_ids = tuple(sorted(dataset.class_names))
benign_ids = [class_id for class_id, name in dataset.class_names.items() if str(name).casefold() == 'benign']
if len(benign_ids) != 1:
    raise RuntimeError('Expected exactly one Benign broad class; inspect class mapping.')
benign_id = int(benign_ids[0])
attack_source_ids = tuple(class_id for class_id in class_ids if class_id != benign_id)
print('Target ID:', benign_id, 'source attack IDs:', attack_source_ids)


## 5. Declare the fair comparison and output record

The new 0% run is the control. Random rates divide by **all training rows**; targeted rates divide by **eligible attack-source rows excluding Benign**. Thus 5% random and 5% targeted are not equal row budgets. Each model starts from seed 42 with the clean-baseline architecture. Clean validation loss selects the checkpoint; this assumes a trusted clean validation set, not a poisoning defense. Test data is not used for tuning.


In [ ]:
OUTPUT_DIR = Path('/kaggle/working') / ('review2_label_flip_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ'))
OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
model_config = ModelConfig(num_features=len(dataset.feature_names), num_classes=len(class_ids), hidden_dim=64, num_heads=4, num_layers=2, mlp_dim=128, batch_size=128, learning_rate=0.001, epochs=20, early_stopping_patience=5, device='cuda', mixed_precision=False, seed=42)
conditions = [('clean_0', 'random', 0.0)] + [(f'{mode}_{int(rate * 100)}', mode, rate) for mode in ('random', 'targeted') for rate in (0.05, 0.10, 0.20)]
np.savez_compressed(OUTPUT_DIR / 'shared_row_indices.npz', selected_original=sample.row_indices, train_positions=split.train, validation_positions=split.validation, test_positions=split.test, train_original=dataset.train.row_indices, validation_original=dataset.validation.row_indices, test_original=dataset.test.row_indices)
preprocessing = {'feature_names': list(dataset.preprocessor.feature_names), 'imputation_values': dataset.preprocessor.imputation_values.tolist(), 'means': dataset.preprocessor.means.tolist(), 'scales': dataset.preprocessor.scales.tolist(), 'class_names': {str(key): value for key, value in dataset.class_names.items()}}
(OUTPUT_DIR / 'preprocessing.json').write_text(json.dumps(preprocessing, indent=2), encoding='utf-8')
manifest = {'purpose': 'static training-label poisoning on a balanced subset; not continual or replay poisoning', 'source_sha': checked_out_sha, 'environment': environment, 'input': {'path': str(parquet_path), 'bytes': parquet_path.stat().st_size, 'sha256': input_sha256, 'provenance': 'combined flow collection; per-row source provenance unverified'}, 'sampling': {'per_class_limit': 2000, 'seed': 42, 'available_counts': sample.available_counts}, 'split': {'seed': 42, 'fractions': [0.70, 0.15, 0.15], 'shapes': {name: list(getattr(dataset, name).features.shape) for name in ('train', 'validation', 'test')}}, 'model_config': asdict(model_config), 'class_ids': list(class_ids), 'target_class_id': benign_id, 'source_class_ids': list(attack_source_ids), 'model_selection': 'minimum clean validation loss; trusted clean validation set; test not used for selection', 'timings_seconds': {'checksum': checksum_seconds, 'sampling': sampling_seconds, 'split_and_preprocessing': preparation_seconds}, 'conditions': {}}
print('Conditions:', conditions, 'output:', OUTPUT_DIR)


## 6. Train and audit every condition

Only copied training labels are changed. The same feature matrices, clean validation/test labels, architecture, initialization seed, and split are reused. Each `fit` starts from a fresh model; its best clean-validation-loss weights are restored. We assert the initial-weight hash is identical across conditions. Files in separate condition folders preserve positions **and original dataset row indices** of changed labels, budget denominators, histories, metrics, timings, checkpoints, and reload checks. Confidence flags on all-known test classes are diagnostics only.


In [ ]:
condition_results = {}
initial_weight_hash = None
for condition_name, mode, rate in conditions:
    folder = OUTPUT_DIR / condition_name
    folder.mkdir()
    options = {'mode': mode, 'allowed_class_ids': class_ids, 'original_row_indices': dataset.train.row_indices}
    if mode == 'targeted':
        options.update(source_class_ids=attack_source_ids, target_class_id=benign_id)
    attack = apply_label_flip(dataset.train.labels, rate, 42, **options)
    assert np.array_equal(dataset.train.labels[attack.changed_indices] != attack.poisoned_labels[attack.changed_indices], np.ones(attack.changed_count, dtype=bool))
    assert np.array_equal(attack.changed_original_row_indices, dataset.train.row_indices[attack.changed_indices])
    model = TabularTransformerClassifier(model_config, class_ids=class_ids)
    digest = hashlib.sha256()
    for key, value in sorted(model.network.state_dict().items()):
        digest.update(key.encode()); digest.update(value.detach().cpu().numpy().tobytes())
    current_initial_hash = digest.hexdigest()
    if initial_weight_hash is None:
        initial_weight_hash = current_initial_hash
    assert current_initial_hash == initial_weight_hash, 'Initial weights differ across conditions'
    torch.cuda.synchronize(); start = perf_counter()
    history = model.fit(dataset.train.features, attack.poisoned_labels, dataset.validation.features, dataset.validation.labels, verbose=True)
    torch.cuda.synchronize(); training_seconds = perf_counter() - start
    assert history.best_epoch == int(np.argmin(history.validation_loss)) + 1
    assert np.isfinite(history.train_loss + history.validation_loss + history.validation_accuracy).all()
    start = perf_counter()
    test_predictions = model.predict(dataset.test.features)
    probabilities = model.predict_proba(dataset.test.features)
    assert probabilities.shape == (len(dataset.test.labels), len(model.class_ids))
    assert np.isfinite(probabilities).all() and np.allclose(probabilities.sum(axis=1), 1, atol=1e-5)
    assert np.array_equal(test_predictions, np.asarray(model.class_ids)[probabilities.argmax(axis=1)])
    metrics = classification_metrics(dataset.test.labels, test_predictions, class_ids=model.class_ids, class_names=dataset.class_names, benign_class_id=benign_id, source_class_ids=attack_source_ids, target_class_id=benign_id)
    confidence_flag_counts = {'threshold': 0.5, 'flagged': int(detect_unknown(probabilities, 0.5).sum()), 'total_known_test_rows': len(dataset.test.labels)}
    evaluation_seconds = perf_counter() - start
    checkpoint = folder / 'best_validation.pt'
    model.save(checkpoint)
    reloaded = TabularTransformerClassifier.load(checkpoint, device='cuda')
    reload_identical = reloaded.class_ids == model.class_ids and np.array_equal(reloaded.predict(dataset.test.features), test_predictions)
    assert reload_identical, 'Checkpoint reload predictions differ'
    np.savez_compressed(folder / 'changed_indices.npz', train_positions=attack.changed_indices, original_dataset_rows=attack.changed_original_row_indices, before=attack.original_labels[attack.changed_indices], after=attack.poisoned_labels[attack.changed_indices])
    attack_record = {'mode': attack.mode, 'requested_rate': attack.requested_rate, 'seed': attack.seed, 'budget_denominator': attack.budget_denominator, 'eligible_count': attack.eligible_count, 'changed_count': attack.changed_count, 'achieved_eligible_fraction': attack.achieved_eligible_fraction, 'achieved_total_fraction': attack.achieved_total_fraction, 'allowed_class_ids': list(attack.allowed_class_ids), 'source_class_ids': list(attack.source_class_ids) if attack.source_class_ids is not None else None, 'target_class_id': attack.target_class_id, 'rounding': 'floor(rate * eligible_count + 0.5)'}
    (folder / 'attack.json').write_text(json.dumps(attack_record, indent=2), encoding='utf-8')
    (folder / 'history.json').write_text(json.dumps(history.to_dict(), indent=2), encoding='utf-8')
    (folder / 'metrics.json').write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    condition_results[condition_name] = {'attack': attack_record, 'metrics': metrics, 'history': history.to_dict(), 'timings_seconds': {'training': training_seconds, 'evaluation': evaluation_seconds}, 'initial_weight_sha256': current_initial_hash, 'checkpoint_reload_identical': reload_identical, 'confidence_flag_counts_only': confidence_flag_counts, 'checkpoint': str(checkpoint.relative_to(OUTPUT_DIR))}
    manifest['conditions'][condition_name] = condition_results[condition_name]
    (OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
    print(condition_name, 'changed:', attack.changed_count, '/', attack.eligible_count, 'eligible,', round(attack.achieved_total_fraction, 4), 'of all training rows; accuracy:', metrics['accuracy'], 'macro-F1:', metrics['macro_f1'], 'benign FPR:', metrics['benign_false_positive']['rate'], 'source-to-Benign:', metrics['source_to_target_misclassification']['rate'])


## 7. Compare each attack against the newly trained 0% control

Report deltas from this run's clean control, not from the earlier clean-baseline artifact. Source-to-Benign errors also exist in a clean model: their difference is an association in this controlled experiment, not proof that every individual error was caused by poisoning. Random and targeted percentages use different denominators. Send the generated manifest, per-condition metrics/histories, notebook output, and plots or confusion matrices back for review; do not commit bulk Kaggle artifacts.


In [ ]:
control = condition_results['clean_0']['metrics']
comparison = {}
for name, result in condition_results.items():
    metrics = result['metrics']
    comparison[name] = {'mode': result['attack']['mode'], 'requested_rate': result['attack']['requested_rate'], 'budget_denominator': result['attack']['budget_denominator'], 'changed_count': result['attack']['changed_count'], 'eligible_count': result['attack']['eligible_count'], 'achieved_total_fraction': result['attack']['achieved_total_fraction'], 'accuracy': metrics['accuracy'], 'accuracy_delta_vs_control': metrics['accuracy'] - control['accuracy'], 'macro_f1': metrics['macro_f1'], 'macro_f1_delta_vs_control': metrics['macro_f1'] - control['macro_f1'], 'benign_false_positive_rate': metrics['benign_false_positive']['rate'], 'benign_fpr_delta_vs_control': metrics['benign_false_positive']['rate'] - control['benign_false_positive']['rate'], 'source_to_target_rate': metrics['source_to_target_misclassification']['rate'], 'source_to_target_delta_vs_control': metrics['source_to_target_misclassification']['rate'] - control['source_to_target_misclassification']['rate'], 'per_class_recall_f1': {key: {'name': value['name'], 'recall': value['recall'], 'f1': value['f1']} for key, value in metrics['per_class'].items()}}
(OUTPUT_DIR / 'comparison.json').write_text(json.dumps(comparison, indent=2), encoding='utf-8')
manifest['comparison_file'] = 'comparison.json'
(OUTPUT_DIR / 'manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
display(pd.DataFrame([{key: value for key, value in row.items() if key != 'per_class_recall_f1'} | {'condition': name} for name, row in comparison.items()]).set_index('condition'))
for name, row in comparison.items():
    print(name, 'per-class recall/F1:', row['per_class_recall_f1'])
print('Output folder:', OUTPUT_DIR, 'source SHA:', checked_out_sha, 'initial weights:', initial_weight_hash)
